In [1]:
import pandas as pd

In [2]:
england_all = pd.read_csv(
    "data/processed/england_ae_2019_to_2026.csv",
    parse_dates=["period"]
)

print(f"Rows: {len(england_all):,}")
print(f"Columns: {list(england_all.columns)}")
print(f"Date range: {england_all['period'].min().strftime('%Y-%m')} to {england_all['period'].max().strftime('%Y-%m')}")
print(f"Unique periods: {england_all['period'].nunique()}")
print(f"Period dtype: {england_all['period'].dtype}")
england_all.head(3)

Rows: 10,736
Columns: ['trust_code', 'trust_region', 'trust_name', 'attendances_type1', 'breaches_4hr_type1', 'pct_4hr_type1', 'waits_12hr', 'admissions_type1', 'total_admissions', 'period']
Date range: 2019-04 to 2026-05
Unique periods: 86
Period dtype: datetime64[us]


,trust_code,trust_region,trust_name,attendances_type1,breaches_4hr_type1,pct_4hr_type1,waits_12hr,admissions_type1,total_admissions,period
0,RDD,NHS England East Of England,Basildon And Thurrock University Hospitals NHS...,11354.0,811.0,0.928571,0.0,3098.0,3360.0,2019-04-01
1,RC1,NHS England East Of England,Bedford Hospital NHS Trust,6349.0,1334.0,0.789888,0.0,2069.0,2328.0,2019-04-01
2,RGT,NHS England East Of England,Cambridge University Hospitals NHS Foundation ...,10412.0,2520.0,0.757972,0.0,3426.0,4013.0,2019-04-01


In [3]:
import requests

In [ ]:

RESOURCE_ID = "37ba17b1-c323-492c-87d5-e986aae9ab59"
URL = "https://www.opendata.nhs.scot/api/3/action/datastore_search"

# Peek first — 5 rows only
params = {"resource_id": RESOURCE_ID, "limit": 5}
response = requests.get(URL, params=params)

print(f"Status: {response.status_code}")
print(f"Success: {response.json().get('success')}")

records = response.json()["result"]["records"]
total_rows = response.json()["result"]["total"]
peek_df = pd.DataFrame(records)

print(f"\nTotal rows available: {total_rows:,}")
print(f"Columns: {list(peek_df.columns)}")
peek_df

Status: 200
Success: True

Total rows available: 39,422
Columns: ['_id', 'Month', 'Country', 'HBT', 'TreatmentLocation', 'DepartmentType', 'AttendanceCategory', 'NumberOfAttendancesAll', 'NumberWithin4HoursAll', 'NumberOver4HoursAll', 'PercentageWithin4HoursAll', 'NumberOfAttendancesEpisode', 'NumberOfAttendancesEpisodeQF', 'NumberWithin4HoursEpisode', 'NumberWithin4HoursEpisodeQF', 'NumberOver4HoursEpisode', 'NumberOver4HoursEpisodeQF', 'PercentageWithin4HoursEpisode', 'PercentageWithin4HoursEpisodeQF', 'NumberOver8HoursEpisode', 'NumberOver8HoursEpisodeQF', 'PercentageOver8HoursEpisode', 'PercentageOver8HoursEpisodeQF', 'NumberOver12HoursEpisode', 'NumberOver12HoursEpisodeQF', 'PercentageOver12HoursEpisode', 'PercentageOver12HoursEpisodeQF']


,_id,Month,Country,HBT,TreatmentLocation,DepartmentType,AttendanceCategory,NumberOfAttendancesAll,NumberWithin4HoursAll,NumberOver4HoursAll,...,PercentageWithin4HoursEpisode,PercentageWithin4HoursEpisodeQF,NumberOver8HoursEpisode,NumberOver8HoursEpisodeQF,PercentageOver8HoursEpisode,PercentageOver8HoursEpisodeQF,NumberOver12HoursEpisode,NumberOver12HoursEpisodeQF,PercentageOver12HoursEpisode,PercentageOver12HoursEpisodeQF
0,1,200707,S92000003,S08000015,A101H,Type 3,Unplanned,252,252,0,...,NaN,z,NaN,z,NaN,z,NaN,z,NaN,z
1,2,200707,S92000003,S08000015,A101H,Type 3,All,252,252,0,...,NaN,z,NaN,z,NaN,z,NaN,z,NaN,z
2,3,200707,S92000003,S08000015,A111H,Type 1,Unplanned,5414,5290,124,...,97.7,,26.0,,0.5,,24.0,,0.4,
3,4,200707,S92000003,S08000015,A111H,Type 1,All,5414,5290,124,...,97.7,,26.0,,0.5,,24.0,,0.4,
4,5,200707,S92000003,S08000015,A207H,Type 3,Unplanned,92,92,0,...,NaN,z,NaN,z,NaN,z,NaN,z,NaN,z


In [10]:
import requests
import pandas as pd
import time

RESOURCE_ID = "37ba17b1-c323-492c-87d5-e986aae9ab59"
URL = "https://www.opendata.nhs.scot/api/3/action/datastore_search"
BATCH_SIZE = 10000  # well under CKAN's 32k cap, safe margin

# Step 1 — find out how many rows we're expecting in total
first_response = requests.get(URL, params={"resource_id": RESOURCE_ID, "limit": 1})
total_rows = first_response.json()["result"]["total"]
print(f"Total rows on server: {total_rows:,}")
print(f"Will fetch in batches of {BATCH_SIZE:,}\n")

# Step 2 — pull in batches
all_batches = []
offset = 0

while offset < total_rows:
    params = {
        "resource_id": RESOURCE_ID,
        "limit": BATCH_SIZE,
        "offset": offset
    }
    response = requests.get(URL, params=params)
    
    if response.status_code != 200:
        raise Exception(f"API failed at offset {offset}: {response.status_code}")
    
    batch_records = response.json()["result"]["records"]
    all_batches.append(pd.DataFrame(batch_records))
    
    print(f"  Pulled rows {offset:,} to {offset + len(batch_records):,} "
          f"({len(batch_records):,} rows)")
    
    offset += BATCH_SIZE
    time.sleep(0.5)  # be polite to PHS's server

# Step 3 — combine into one dataframe
scotland_raw = pd.concat(all_batches, ignore_index=True)

# Step 4 — verify we got everything (the count-check we discussed)
print(f"\n=== VERIFICATION ===")
print(f"Expected rows: {total_rows:,}")
print(f"Actual rows:   {len(scotland_raw):,}")
print(f"Match: {'✓' if len(scotland_raw) == total_rows else '✗ MISMATCH'}")
print(f"Columns: {len(scotland_raw.columns)}")

Total rows on server: 39,422
Will fetch in batches of 10,000

  Pulled rows 0 to 10,000 (10,000 rows)
  Pulled rows 10,000 to 20,000 (10,000 rows)
  Pulled rows 20,000 to 30,000 (10,000 rows)
  Pulled rows 30,000 to 39,422 (9,422 rows)

=== VERIFICATION ===
Expected rows: 39,422
Actual rows:   39,422
Match: ✓
Columns: 27


In [11]:
from pathlib import Path
from datetime import date

# Make the raw scotland folder if it doesn't exist
scotland_raw_folder = Path("data/raw/scotland")
scotland_raw_folder.mkdir(parents=True, exist_ok=True)

# Filename includes today's date — this is the snapshot date
today = date.today().strftime("%Y%m%d")
snapshot_path = scotland_raw_folder / f"phs_monthly_ae_raw_{today}.csv"

# Save
scotland_raw.to_csv(snapshot_path, index=False)

print(f"✓ Snapshot saved")
print(f"  Path: {snapshot_path}")
print(f"  Rows: {len(scotland_raw):,}")
print(f"  Size: {snapshot_path.stat().st_size / 1024:.1f} KB")

✓ Snapshot saved
  Path: data\raw\scotland\phs_monthly_ae_raw_20260624.csv
  Rows: 39,422
  Size: 4158.5 KB


In [12]:
# Step 0 — start with the raw data, leave it untouched
df = scotland_raw.copy()
print(f"Starting rows: {len(df):,}")

# Step 1 — filter to AttendanceCategory == "All"
df = df[df["AttendanceCategory"] == "All"]
print(f"After 'All' filter:        {len(df):,} rows")

# Step 2 — filter to DepartmentType == "Type 1"
df = df[df["DepartmentType"] == "Type 1"]
print(f"After Type 1 filter:       {len(df):,} rows")

# Step 3 — filter to the project window (April 2019 → May 2026)
#  Month is a YYYYMM integer-or-string, e.g. 201904 = April 2019
df["Month"] = df["Month"].astype(int)  # ensure numeric for comparison
df = df[(df["Month"] >= 201904) & (df["Month"] <= 202605)]
print(f"After date window filter:  {len(df):,} rows")

print(f"\nFinal shape: {df.shape}")
print(f"Unique boards (HBT): {df['HBT'].nunique()}")
print(f"Unique sites (TreatmentLocation): {df['TreatmentLocation'].nunique()}")
print(f"Unique months: {df['Month'].nunique()}")

Starting rows: 39,422
After 'All' filter:        18,717 rows
After Type 1 filter:       6,994 rows
After date window filter:  2,550 rows

Final shape: (2550, 27)
Unique boards (HBT): 14
Unique sites (TreatmentLocation): 30
Unique months: 85


In [13]:
# Build the full expected month list
import pandas as pd

expected_months = pd.date_range("2019-04-01", "2026-05-01", freq="MS")
expected_yyyymm = set(int(d.strftime("%Y%m")) for d in expected_months)

# What we actually have
actual_yyyymm = set(df["Month"].unique())

# What's missing
missing = expected_yyyymm - actual_yyyymm
extra = actual_yyyymm - expected_yyyymm

print(f"Expected months: {len(expected_yyyymm)}")
print(f"Actual months:   {len(actual_yyyymm)}")
print(f"\nMissing from Scotland data: {sorted(missing)}")
print(f"Extra (shouldn't be there):  {sorted(extra)}")

Expected months: 86
Actual months:   85

Missing from Scotland data: [202605]
Extra (shouldn't be there):  []


In [14]:
# Scotland NHS Board codes — sourced from PHS Geography codes (S08000xxx)
SCOTLAND_BOARD_NAMES = {
    "S08000015": "NHS Ayrshire & Arran",
    "S08000016": "NHS Borders",
    "S08000017": "NHS Dumfries & Galloway",
    "S08000019": "NHS Forth Valley",
    "S08000020": "NHS Grampian",
    "S08000022": "NHS Highland",
    "S08000024": "NHS Lothian",
    "S08000025": "NHS Orkney",
    "S08000026": "NHS Shetland",
    "S08000028": "NHS Western Isles",
    "S08000029": "NHS Fife",
    "S08000030": "NHS Tayside",
    "S08000031": "NHS Greater Glasgow & Clyde",
    "S08000032": "NHS Lanarkshire",
}

# Quick sanity check — should be 14
print(f"Boards in lookup: {len(SCOTLAND_BOARD_NAMES)}")

# Verify the codes match what's in our data
codes_in_data = set(df["HBT"].unique())
codes_in_lookup = set(SCOTLAND_BOARD_NAMES.keys())

missing_from_lookup = codes_in_data - codes_in_lookup
unused_in_lookup = codes_in_lookup - codes_in_data

print(f"Codes in filtered data: {len(codes_in_data)}")
print(f"Missing from lookup (problem): {missing_from_lookup}")
print(f"In lookup but not in data:     {unused_in_lookup}")

Boards in lookup: 14
Codes in filtered data: 14
Missing from lookup (problem): set()
In lookup but not in data:     set()


In [15]:
import pandas as pd

# Test on the first 5 rows before transforming everything
sample = df.head(5).copy()

# Convert YYYYMM integer to datetime
# Approach: cast to string, then parse with the format string
sample["period"] = pd.to_datetime(
    sample["Month"].astype(str),
    format="%Y%m"
)

# Inspect
print("Test conversion on 5 sample rows:")
print(sample[["Month", "period"]])
print(f"\nDtype of period: {sample['period'].dtype}")

Test conversion on 5 sample rows:
        Month     period
25779  201904 2019-04-01
25781  201904 2019-04-01
25783  201904 2019-04-01
26243  201904 2019-04-01
26245  201904 2019-04-01

Dtype of period: datetime64[us]


In [16]:
import pandas as pd
import numpy as np

def transform_scotland_ae(df_filtered: pd.DataFrame, board_lookup: dict) -> pd.DataFrame:
    """
    Transform filtered PHS A&E data into the England-matched schema.
    
    Expects df_filtered to already have these filters applied:
      - AttendanceCategory == "All"
      - DepartmentType == "Type 1"
      - Month within project window
    
    Returns a dataframe with the same columns as england_all.
    """
    
    # ---- Safety check 1: required source columns exist ----
    required_cols = [
        "Month", "HBT", "NumberOfAttendancesAll",
        "NumberOver4HoursAll", "PercentageWithin4HoursAll",
        "NumberOver12HoursEpisode"
    ]
    missing = [c for c in required_cols if c not in df_filtered.columns]
    if missing:
        raise ValueError(f"Scotland transform: missing source columns {missing}")
    
    # ---- Safety check 2: filters were applied (sanity-checking caller) ----
    if "AttendanceCategory" in df_filtered.columns:
        unique_cats = df_filtered["AttendanceCategory"].unique()
        if not (len(unique_cats) == 1 and unique_cats[0] == "All"):
            raise ValueError(
                f"Scotland transform: expected only 'All' category, got {unique_cats}"
            )
    
    # ---- Work on a copy so we don't mutate the input ----
    df = df_filtered.copy()
    
    # ---- Build the clean columns ----
    df["period"] = pd.to_datetime(df["Month"].astype(str), format="%Y%m")
    df["trust_code"] = df["HBT"]
    df["trust_name"] = df["HBT"].map(board_lookup)
    df["trust_region"] = "Scotland"
    df["attendances_type1"] = pd.to_numeric(df["NumberOfAttendancesAll"], errors="coerce")
    df["breaches_4hr_type1"] = pd.to_numeric(df["NumberOver4HoursAll"], errors="coerce")
    df["pct_4hr_type1"] = pd.to_numeric(df["PercentageWithin4HoursAll"], errors="coerce")
    df["waits_12hr"] = pd.to_numeric(df["NumberOver12HoursEpisode"], errors="coerce")
    df["admissions_type1"] = np.nan
    df["total_admissions"] = np.nan
    
    # ---- Select the final columns in the same order as England ----
    final_cols = [
        "trust_code", "trust_region", "trust_name",
        "attendances_type1", "breaches_4hr_type1", "pct_4hr_type1",
        "waits_12hr", "admissions_type1", "total_admissions",
        "period"
    ]
    df = df[final_cols].reset_index(drop=True)
    
    # ---- Safety check 3: no board codes failed the lookup ----
    unmapped = df[df["trust_name"].isna()]["trust_code"].unique()
    if len(unmapped) > 0:
        raise ValueError(f"Scotland transform: board codes not in lookup: {unmapped}")
    
    return df

In [17]:
scotland_clean = transform_scotland_ae(df, SCOTLAND_BOARD_NAMES)

print(f"Shape: {scotland_clean.shape}")
print(f"Columns: {list(scotland_clean.columns)}")
print(f"\nDtypes:")
print(scotland_clean.dtypes)
print(f"\nFirst 3 rows:")
scotland_clean.head(3)

Shape: (2550, 10)
Columns: ['trust_code', 'trust_region', 'trust_name', 'attendances_type1', 'breaches_4hr_type1', 'pct_4hr_type1', 'waits_12hr', 'admissions_type1', 'total_admissions', 'period']

Dtypes:
trust_code                       str
trust_region                     str
trust_name                       str
attendances_type1              int64
breaches_4hr_type1             int64
pct_4hr_type1                float64
waits_12hr                   float64
admissions_type1             float64
total_admissions             float64
period                datetime64[us]
dtype: object

First 3 rows:


,trust_code,trust_region,trust_name,attendances_type1,breaches_4hr_type1,pct_4hr_type1,waits_12hr,admissions_type1,total_admissions,period
0,S08000019,Scotland,NHS Forth Valley,5740,591,89.7,0.0,NaN,NaN,2019-04-01
1,S08000020,Scotland,NHS Grampian,5000,600,88.0,3.0,NaN,NaN,2019-04-01
2,S08000020,Scotland,NHS Grampian,1455,26,98.2,0.0,NaN,NaN,2019-04-01


In [18]:
print(f"Total rows: {len(scotland_clean):,}")
print(f"Unique boards: {scotland_clean['trust_code'].nunique()}")
print(f"Unique sites (rows aren't site-level here, but date+code combos):")
print(f"  Date range: {scotland_clean['period'].min()} to {scotland_clean['period'].max()}")
print(f"  Unique periods: {scotland_clean['period'].nunique()}")
print(f"\nNational headline metrics (across all rows, all sites, all months):")
print(f"  Average Type 1 4-hour %: {scotland_clean['pct_4hr_type1'].mean():.1f}%")
print(f"  Total Type 1 attendances: {scotland_clean['attendances_type1'].sum():,}")
print(f"  Total 12-hour waits: {scotland_clean['waits_12hr'].sum():,.0f}")

Total rows: 2,550
Unique boards: 14
Unique sites (rows aren't site-level here, but date+code combos):
  Date range: 2019-04-01 00:00:00 to 2026-04-01 00:00:00
  Unique periods: 85

National headline metrics (across all rows, all sites, all months):
  Average Type 1 4-hour %: 78.7%
  Total Type 1 attendances: 9,392,841
  Total 12-hour waits: 316,211


In [19]:
from pathlib import Path

processed_folder = Path("data/processed")
processed_folder.mkdir(parents=True, exist_ok=True)

scotland_clean.to_csv("data/processed/scotland_ae_2019_to_2026.csv", index=False)
print(f"Saved {len(scotland_clean):,} rows to data/processed/scotland_ae_2019_to_2026.csv")

Saved 2,550 rows to data/processed/scotland_ae_2019_to_2026.csv


In [20]:
import pandas as pd

scotland = pd.read_csv("data/processed/scotland_ae_2019_to_2026.csv", parse_dates=["period"])

print(f"Rows: {len(scotland):,}")
print(f"Columns: {list(scotland.columns)}")
scotland.head(10)

Rows: 2,550
Columns: ['trust_code', 'trust_region', 'trust_name', 'attendances_type1', 'breaches_4hr_type1', 'pct_4hr_type1', 'waits_12hr', 'admissions_type1', 'total_admissions', 'period']


,trust_code,trust_region,trust_name,attendances_type1,breaches_4hr_type1,pct_4hr_type1,waits_12hr,admissions_type1,total_admissions,period
0,S08000019,Scotland,NHS Forth Valley,5740,591,89.7,0.0,NaN,NaN,2019-04-01
1,S08000020,Scotland,NHS Grampian,5000,600,88.0,3.0,NaN,NaN,2019-04-01
2,S08000020,Scotland,NHS Grampian,1455,26,98.2,0.0,NaN,NaN,2019-04-01
3,S08000015,Scotland,NHS Ayrshire & Arran,6382,717,88.8,85.0,NaN,NaN,2019-04-01
4,S08000015,Scotland,NHS Ayrshire & Arran,3445,561,83.7,51.0,NaN,NaN,2019-04-01
5,S08000016,Scotland,NHS Borders,2774,156,94.4,0.0,NaN,NaN,2019-04-01
6,S08000017,Scotland,NHS Dumfries & Galloway,1208,73,94.0,2.0,NaN,NaN,2019-04-01
7,S08000017,Scotland,NHS Dumfries & Galloway,2958,221,92.5,1.0,NaN,NaN,2019-04-01
8,S08000029,Scotland,NHS Fife,5818,429,92.6,0.0,NaN,NaN,2019-04-01
9,S08000020,Scotland,NHS Grampian,2242,104,95.4,0.0,NaN,NaN,2019-04-01


In [27]:
import pandas as pd

# The RAW snapshot — before any cleaning, filtering, transforming
scotland_raw = pd.read_csv("data/raw/scotland/phs_monthly_ae_raw_20260624.csv")

print(f"Rows: {len(scotland_raw):,}")
print(f"Columns ({len(scotland_raw.columns)}):")
for col in scotland_raw.columns:
    print(f"  - {col}")
print(f"\nFirst 5 rows:")
scotland_raw.head(10)

Rows: 39,422
Columns (27):
  - _id
  - Month
  - Country
  - HBT
  - TreatmentLocation
  - DepartmentType
  - AttendanceCategory
  - NumberOfAttendancesAll
  - NumberWithin4HoursAll
  - NumberOver4HoursAll
  - PercentageWithin4HoursAll
  - NumberOfAttendancesEpisode
  - NumberOfAttendancesEpisodeQF
  - NumberWithin4HoursEpisode
  - NumberWithin4HoursEpisodeQF
  - NumberOver4HoursEpisode
  - NumberOver4HoursEpisodeQF
  - PercentageWithin4HoursEpisode
  - PercentageWithin4HoursEpisodeQF
  - NumberOver8HoursEpisode
  - NumberOver8HoursEpisodeQF
  - PercentageOver8HoursEpisode
  - PercentageOver8HoursEpisodeQF
  - NumberOver12HoursEpisode
  - NumberOver12HoursEpisodeQF
  - PercentageOver12HoursEpisode
  - PercentageOver12HoursEpisodeQF

First 5 rows:


,_id,Month,Country,HBT,TreatmentLocation,DepartmentType,AttendanceCategory,NumberOfAttendancesAll,NumberWithin4HoursAll,NumberOver4HoursAll,...,PercentageWithin4HoursEpisode,PercentageWithin4HoursEpisodeQF,NumberOver8HoursEpisode,NumberOver8HoursEpisodeQF,PercentageOver8HoursEpisode,PercentageOver8HoursEpisodeQF,NumberOver12HoursEpisode,NumberOver12HoursEpisodeQF,PercentageOver12HoursEpisode,PercentageOver12HoursEpisodeQF
0,1,200707,S92000003,S08000015,A101H,Type 3,Unplanned,252,252,0,...,NaN,z,NaN,z,NaN,z,NaN,z,NaN,z
1,2,200707,S92000003,S08000015,A101H,Type 3,All,252,252,0,...,NaN,z,NaN,z,NaN,z,NaN,z,NaN,z
2,3,200707,S92000003,S08000015,A111H,Type 1,Unplanned,5414,5290,124,...,97.7,NaN,26.0,NaN,0.5,NaN,24.0,NaN,0.4,NaN
3,4,200707,S92000003,S08000015,A111H,Type 1,All,5414,5290,124,...,97.7,NaN,26.0,NaN,0.5,NaN,24.0,NaN,0.4,NaN
4,5,200707,S92000003,S08000015,A207H,Type 3,Unplanned,92,92,0,...,NaN,z,NaN,z,NaN,z,NaN,z,NaN,z
5,6,200707,S92000003,S08000015,A207H,Type 3,All,92,92,0,...,NaN,z,NaN,z,NaN,z,NaN,z,NaN,z
6,7,200707,S92000003,S08000015,A210H,Type 1,Unplanned,3530,3355,175,...,95.0,NaN,3.0,NaN,0.1,NaN,1.0,NaN,0.0,NaN
7,8,200707,S92000003,S08000015,A210H,Type 1,All,3530,3355,175,...,95.0,NaN,3.0,NaN,0.1,NaN,1.0,NaN,0.0,NaN
8,9,200707,S92000003,S08000016,B103H,Type 3,Unplanned,20,20,0,...,NaN,z,NaN,z,NaN,z,NaN,z,NaN,z
9,10,200707,S92000003,S08000016,B103H,Type 3,All,20,20,0,...,NaN,z,NaN,z,NaN,z,NaN,z,NaN,z


In [26]:
scotland_raw.columns.to_list()

['_id',
 'Month',
 'Country',
 'HBT',
 'TreatmentLocation',
 'DepartmentType',
 'AttendanceCategory',
 'NumberOfAttendancesAll',
 'NumberWithin4HoursAll',
 'NumberOver4HoursAll',
 'PercentageWithin4HoursAll',
 'NumberOfAttendancesEpisode',
 'NumberOfAttendancesEpisodeQF',
 'NumberWithin4HoursEpisode',
 'NumberWithin4HoursEpisodeQF',
 'NumberOver4HoursEpisode',
 'NumberOver4HoursEpisodeQF',
 'PercentageWithin4HoursEpisode',
 'PercentageWithin4HoursEpisodeQF',
 'NumberOver8HoursEpisode',
 'NumberOver8HoursEpisodeQF',
 'PercentageOver8HoursEpisode',
 'PercentageOver8HoursEpisodeQF',
 'NumberOver12HoursEpisode',
 'NumberOver12HoursEpisodeQF',
 'PercentageOver12HoursEpisode',
 'PercentageOver12HoursEpisodeQF']

In [28]:
import requests
import pandas as pd

# The A&E Sites lookup dataset
SITES_RESOURCE_ID = "1a4e3f48-3d9b-4769-80e9-3ef6d27852fe"
URL = "https://www.opendata.nhs.scot/api/3/action/datastore_search"

# It's a small table — one call, generous limit, should get everything
params = {"resource_id": SITES_RESOURCE_ID, "limit": 500}
response = requests.get(URL, params=params)

print(f"Status: {response.status_code}")
print(f"Success: {response.json().get('success')}")

records = response.json()["result"]["records"]
total_rows = response.json()["result"]["total"]
sites_raw = pd.DataFrame(records)

print(f"\nTotal rows on server: {total_rows}")
print(f"Rows pulled: {len(sites_raw)}")
print(f"Columns: {list(sites_raw.columns)}")
print(f"\nFirst 5 rows:")
sites_raw.head()

Status: 200
Success: True

Total rows on server: 102
Rows pulled: 102
Columns: ['_id', 'HB', 'TreatmentLocationName', 'TreatmentLocationCode', 'TreatmentLocationPostcode', 'CurrentDepartmentType', 'FileType', 'Comments', 'Status']

First 5 rows:


,_id,HB,TreatmentLocationName,TreatmentLocationCode,TreatmentLocationPostcode,CurrentDepartmentType,FileType,Comments,Status
0,1,S08000015,University Hospital Ayr,A210H,KA6 6DX,Type 1,Episode,None,Open
1,2,S08000015,University Hospital Crosshouse,A111H,KA2 0BE,Type 1,Episode,None,Open
2,3,S08000015,Arran War Memorial Hospital,A101H,KA278LF,Type 3,Aggregate,None,Open
3,4,S08000015,Davidson Cottage Hospital,A207H,KA269DS,Type 3,Aggregate,Closed from May 2010,Closed
4,5,S08000015,Girvan Community Hospital,A216H,KA269HQ,Type 3,Aggregate,Opened from May 2010,Open


In [29]:
# How many sites by department type?
print("Sites by department type:")
print(sites_raw["CurrentDepartmentType"].value_counts())
print()
print("Sites by status:")
print(sites_raw["Status"].value_counts())
print()
# Just Type 1 that are currently open
type1_open = sites_raw[
    (sites_raw["CurrentDepartmentType"] == "Type 1") & 
    (sites_raw["Status"] == "Open")
]
print(f"Type 1 sites currently open: {len(type1_open)}")

Sites by department type:
CurrentDepartmentType
Type 3    72
Type 1    30
Name: count, dtype: int64

Sites by status:
Status
Open      61
Closed    41
Name: count, dtype: int64

Type 1 sites currently open: 30


In [30]:
# Build the sites lookup — only Type 1, only the columns we need
sites_lookup = (
    sites_raw
    .loc[sites_raw["CurrentDepartmentType"] == "Type 1"]
    [["TreatmentLocationCode", "TreatmentLocationName", "HB"]]
    .rename(columns={
        "TreatmentLocationCode": "site_code",
        "TreatmentLocationName": "site_name",
        "HB": "board_code"
    })
    .reset_index(drop=True)
)

print(f"Sites lookup shape: {sites_lookup.shape}")
print(f"Unique sites: {sites_lookup['site_code'].nunique()}")
print(f"\nFirst 10 rows:")
sites_lookup.head(10)

Sites lookup shape: (30, 3)
Unique sites: 30

First 10 rows:


,site_code,site_name,board_code
0,A210H,University Hospital Ayr,S08000015
1,A111H,University Hospital Crosshouse,S08000015
2,B120H,Borders General Hospital,S08000016
3,Y146H,Dumfries & Galloway Royal Infirmary,S08000017
4,Y144H,Galloway Community Hospital,S08000017
5,F704H,Victoria Hospital (NHS Fife),S08000029
6,V217H,Forth Valley Royal Hospital,S08000019
7,N101H,Aberdeen Royal Infirmary,S08000020
8,N411H,Dr Gray's Hospital,S08000020
9,N121H,Royal Aberdeen Children's Hospital,S08000020


In [31]:
# Start from the filtered raw Scotland data (before we did the muddled transform)
# 'df' should still be your filtered Type 1, All, in-window dataframe (2,550 rows)

# Attach hospital name and board code from the lookup
scotland_joined = df.merge(
    sites_lookup,
    left_on="TreatmentLocation",   # activity data's site code column
    right_on="site_code",           # lookup's site code column
    how="left"                      # keep all activity rows, add lookup info where available
)

print(f"Rows after join: {len(scotland_joined):,}")
print(f"Should still be 2,550 — same rows, just extra columns\n")

# Check no rows lost their hospital name
missing_names = scotland_joined["site_name"].isna().sum()
print(f"Rows with missing hospital name: {missing_names}")

# See what we now have
print(f"\nSample of joined data:")
scotland_joined[["Month", "HBT", "TreatmentLocation", "site_name", "NumberOfAttendancesAll"]].head(3)

Rows after join: 2,550
Should still be 2,550 — same rows, just extra columns

Rows with missing hospital name: 0

Sample of joined data:


,Month,HBT,TreatmentLocation,site_name,NumberOfAttendancesAll
0,201904,S08000019,V217H,Forth Valley Royal Hospital,5740
1,201904,S08000020,N101H,Aberdeen Royal Infirmary,5000
2,201904,S08000020,N121H,Royal Aberdeen Children's Hospital,1455


In [33]:
import pandas as pd
import numpy as np

scotland_clean = pd.DataFrame({
    "trust_code": scotland_joined["TreatmentLocation"],
    "trust_region": scotland_joined["HBT"].map(SCOTLAND_BOARD_NAMES),
    "trust_name": scotland_joined["site_name"],
    "attendances_type1": pd.to_numeric(scotland_joined["NumberOfAttendancesAll"], errors="coerce"),
    "breaches_4hr_type1": pd.to_numeric(scotland_joined["NumberOver4HoursAll"], errors="coerce"),
    "pct_4hr_type1": pd.to_numeric(scotland_joined["PercentageWithin4HoursAll"], errors="coerce"),
    "waits_12hr": pd.to_numeric(scotland_joined["NumberOver12HoursEpisode"], errors="coerce"),
    "admissions_type1": np.nan,
    "total_admissions": np.nan,
    "period": pd.to_datetime(scotland_joined["Month"].astype(str), format="%Y%m")
})

print(f"Shape: {scotland_clean.shape}")
print(f"Columns: {list(scotland_clean.columns)}")
print(f"\nFirst 5 rows:")
scotland_clean.head()

Shape: (2550, 10)
Columns: ['trust_code', 'trust_region', 'trust_name', 'attendances_type1', 'breaches_4hr_type1', 'pct_4hr_type1', 'waits_12hr', 'admissions_type1', 'total_admissions', 'period']

First 5 rows:


,trust_code,trust_region,trust_name,attendances_type1,breaches_4hr_type1,pct_4hr_type1,waits_12hr,admissions_type1,total_admissions,period
0,V217H,NHS Forth Valley,Forth Valley Royal Hospital,5740,591,89.7,0.0,NaN,NaN,2019-04-01
1,N101H,NHS Grampian,Aberdeen Royal Infirmary,5000,600,88.0,3.0,NaN,NaN,2019-04-01
2,N121H,NHS Grampian,Royal Aberdeen Children's Hospital,1455,26,98.2,0.0,NaN,NaN,2019-04-01
3,A111H,NHS Ayrshire & Arran,University Hospital Crosshouse,6382,717,88.8,85.0,NaN,NaN,2019-04-01
4,A210H,NHS Ayrshire & Arran,University Hospital Ayr,3445,561,83.7,51.0,NaN,NaN,2019-04-01


In [34]:
scotland_clean.to_csv("data/processed/scotland_ae_2019_to_2026.csv", index=False)
print(f"Saved {len(scotland_clean):,} rows to data/processed/scotland_ae_2019_to_2026.csv")
print("(overwrote the earlier muddled version)")

Saved 2,550 rows to data/processed/scotland_ae_2019_to_2026.csv
(overwrote the earlier muddled version)


In [35]:
import pandas as pd

# ---- Load both source files ----
england = pd.read_csv("data/processed/england_ae_2019_to_2026.csv", parse_dates=["period"])
scotland = pd.read_csv("data/processed/scotland_ae_2019_to_2026.csv", parse_dates=["period"])

print(f"England loaded: {len(england):,} rows")
print(f"Scotland loaded: {len(scotland):,} rows")

# ---- Add country column to each ----
england["country"] = "England"
scotland["country"] = "Scotland"

# ---- Align windows: keep only overlap (April 2019 → April 2026) ----
england_aligned = england[england["period"] <= "2026-04-01"].copy()
scotland_aligned = scotland[scotland["period"] <= "2026-04-01"].copy()

print(f"\nAfter window alignment (both ≤ April 2026):")
print(f"  England: {len(england_aligned):,} rows")
print(f"  Scotland: {len(scotland_aligned):,} rows")

# ---- Stack them into one dataframe ----
combined = pd.concat([england_aligned, scotland_aligned], ignore_index=True)

# ---- Verify ----
print(f"\n=== COMBINED DATASET ===")
print(f"Total rows: {len(combined):,}")
print(f"Rows by country:")
print(combined["country"].value_counts())
print(f"\nDate range: {combined['period'].min().strftime('%Y-%m')} to {combined['period'].max().strftime('%Y-%m')}")
print(f"Unique months: {combined['period'].nunique()}")
print(f"Columns: {list(combined.columns)}")

England loaded: 10,736 rows
Scotland loaded: 2,550 rows

After window alignment (both ≤ April 2026):
  England: 10,615 rows
  Scotland: 2,550 rows

=== COMBINED DATASET ===
Total rows: 13,165
Rows by country:
country
England     10615
Scotland     2550
Name: count, dtype: int64

Date range: 2019-04 to 2026-04
Unique months: 85
Columns: ['trust_code', 'trust_region', 'trust_name', 'attendances_type1', 'breaches_4hr_type1', 'pct_4hr_type1', 'waits_12hr', 'admissions_type1', 'total_admissions', 'period', 'country']


In [36]:
combined.to_csv("data/processed/combined_ae_2019_to_2026.csv", index=False)
print(f"Saved {len(combined):,} rows to data/processed/combined_ae_2019_to_2026.csv")


Saved 13,165 rows to data/processed/combined_ae_2019_to_2026.csv
